# 7 — Visualisations: fields along skewers, spectra, and P1D/ξ1D by order

This notebook follows a toy model from the gas fields to the two-point statistics:
1. **Fields.** HI and Si III number densities are drawn from a 3D Gaussian random field (`PowerLawGRF`). Temperature and velocity are constant.
2. **Skewers and τ.** A 3×3 grid of skewers crosses the box. `OpticalDepthCalculator` computes τ(v) for Lyα and Si III 1206.
3. **Neon renderings.** The fields along the skewers and the spectra are rendered in 3D, with an optional GIF that grows the skewers frame by frame.
4. **Overflux and statistics.** The overflux δ = F/⟨F⟩ − 1 of each line and of the total is computed, and P1D/ξ1D are split by line pair and by order.
5. **Checks.** The GRF reproduces its input power law, and the orders sum to the total.

In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"  # must be set before importing jax
import logging
logging.basicConfig(level=logging.WARNING, format="%(asctime)s - %(levelname)s - %(message)s")

import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

import skewerjax as sjx
import skewerjax.utils.plotting_tools as pt
import gc
from types import SimpleNamespace
from pathlib import Path
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology
from skewerjax.fields.fields import ScalarField, VectorField, Fields
from skewerjax.fields.toy_model_samplers import PowerLawGRF
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux
import skewerjax.optical_depth.line_profile as lp
import skewerjax.optical_depth.optical_depth as od
from skewerjax.optical_depth.overflux import DeltaFieldBuilder
from skewerjax.summary_stats.P1D_xi1D import OverfluxStatistics1D
from skewerjax.summary_stats.P3D_xi3D import OverfluxStatistics3D
from skewerjax.summary_stats.two_point import xi3d_from_power, xi_from_power
from skewerjax.constants.physical_constants import phys

plt.style.use("tableau-colorblind10")
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc("font", **font)
plt.rcParams.update(rcnew)
%matplotlib inline
# %matplotlib widget   # interactive figures

## Cosmology, box and lines

In [ ]:
cosmology = Cosmology(H0=phys.H0, Omega_r_0=0.0, Omega_m_0=1.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
ii_cosmology = 0          # skewers/cosmology support batches; fields and tau use one cosmology

L_box, z_box = 110.7, 2.0                        # [cMpc]
N_grid_skewers, N_ppsk, N_v = 3, 4096, 1024      # skewers per side, points per skewer, velocity pixels
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
colors = pt._neon_colors(line_bundle.species_keys)

## Fields
The number density is n = 10^(log_mid + log_amp·tanh(δ/1.2)), mapped into [n0_min, n0_max]. Here δ is a 3D GRF with P(k) ∝ (k/k₀)^(n_s−1), normalised to unit variance, on a 256³ grid.

Both species use the same realisation and the same temperature T₀. The previous version filtered the spectrum with exp(−(k/50)²). `PowerLawGRF` has no cutoff, but the factor was ≥ 0.98 below the grid Nyquist (k_Nyq = 7.3 cMpc⁻¹), so dropping it makes no visible difference. Because δ is normalised, the amplitude A_s does not change the field.

In [ ]:
T0 = 1.5e4                                       # [K]
n0_min, n0_max = 1e-16, 1e-10                    # [cm^-3]
grf = PowerLawGRF(A_s=5.0, k_0=2.0, n_s=-1.5, L=(L_box,) * 3, N=(256,) * 3)   # (k/k0)^-2.5
number_density_sampler = toy_samplers.number_density_sampler_gaussian_3d(
    jax.random.PRNGKey(0), grf, n0_min=n0_min, n0_max=n0_max, contrast=1.2)

def species_fields(n_sampler):
    return {"temperature": ScalarField(toy_samplers.temperature_sampler(T0, T0, L_box)),     # [K]
            "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),         # [km/s]
            "number_density": ScalarField(n_sampler),                                        # [cm^-3]
            "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),          # [km/s]
            "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0))}        # [GHz]

fields = Fields({sp: species_fields(number_density_sampler) for sp in line_bundle.species_keys})

## GIF of growing skewers (optional)
Each frame truncates the skewers at a different length, recomputes τ, renders all figures with `generate_all_simulation_plots`, and rotates the camera by up to 180° over the animation. The PNG frames go to `SAVE_DIR`, and `create_animated_gifs_and_cleanup` assembles one GIF per figure type and deletes the frames. Use more `FRAMES` (e.g. 128) for a smooth animation.

In [ ]:
MAKE_GIF = True
SAVE_DIR = Path("../SAVED_FIGURES/simulation_plots_evolution")
FRAMES = [N_ppsk // 2, N_ppsk - 1]               # e.g. np.linspace(1, N_ppsk - 1, 128, dtype=int)
AZIM_START = -43.0

if MAKE_GIF:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    points_full = jnp.reshape(aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y"),
                              (N_grid_skewers ** 2, N_ppsk, 3))
    a_full = SkewerBundle(spatial_points=points_full, cosmologies=cosmology, a0=1 / (1 + z_box), x0=L_box).scale_factors
    for frame, ii in enumerate(FRAMES):
        azim = AZIM_START + 180.0 * frame / max(len(FRAMES) - 1, 1)
        n_pts = ii + 1
        a_dagger = float(a_full[0, 0, n_pts - 1])         # observer at the end of the truncated skewer
        sb = SkewerBundle(spatial_points=points_full[:, :n_pts], cosmologies=cosmology, a0=a_dagger,
                          x0=float(points_full[0, n_pts - 1, 1]))
        a_em = float(sb.scale_factors[0, 0, 0])
        v_max = phys.c_kms * jnp.log(float(a_full[0, 0, -1]) / a_em)   # fixed axis: end of the full box
        velocities = jnp.linspace(0, v_max, N_v)
        tau = od.OpticalDepthCalculator(skewer_bundle=sb, velocities_kms=velocities, line_bundle=line_bundle,
                                        fields=fields, a_dagger=a_dagger)
        tau.compute()
        optical_depth = tau.optical_depth
        tau.wrap_to_ref_line()
        figs = pt.generate_all_simulation_plots(
            line_bundle, sb, fields, L_box, velocities,
            lp.velocities_to_line_centered_wavelengths(velocities, line_bundle, a_dagger=a_dagger, a_obs=a_dagger),
            lp.reference_velocities_to_line_centered(velocities, line_bundle, reference_line_idx=0),
            optical_depth, tau.optical_depth, a_dagger, a_em, a_dagger, -jnp.max(line_bundle.delta_v), v_max,
            plot_normalized_velocity_units=False, plot_wrapped_velocity_units=False,
            view_elev=20.0, view_azim=azim, los_range=(0.0, L_box), show=False)
        for idx, fig in enumerate(figs):                  # 0-3 cylinders, 4 box, 5-8 transmittance
            fig.savefig(SAVE_DIR / f"iter{ii:04d}_fig{idx}.png", dpi=128, bbox_inches="tight")
            plt.close(fig)
        del tau, optical_depth, sb, figs
        gc.collect()
        jax.clear_caches()
        print(f"frame {frame}: N_ppsk = {n_pts}, azimuth {azim:.1f} deg")
    pt.create_animated_gifs_and_cleanup(str(SAVE_DIR), duration_ms=500, loop=0, delete_originals=True)

## Skewers and optical depth

In [ ]:
points = jnp.reshape(aux.make_skewer_grid_points(L_box, N_grid_skewers, N_ppsk, direction="y"),
                     (N_grid_skewers ** 2, N_ppsk, 3))
skewer_bundle = SkewerBundle(spatial_points=points, cosmologies=cosmology, a0=1 / (1 + z_box), x0=L_box)

a_dagger, a_obs = 1 / (1 + z_box), 1.0
a_em = skewer_bundle.scale_factors[0, 0, 0]
v_max = phys.c_kms * jnp.log(a_dagger / a_em)
velocities_kms = jnp.linspace(0, v_max, N_v)
dv = float(velocities_kms[1] - velocities_kms[0])

tau = od.OpticalDepthCalculator(skewer_bundle=skewer_bundle, velocities_kms=velocities_kms,
                                line_bundle=line_bundle, fields=fields, a_dagger=a_dagger)
tau.compute()
optical_depth = tau.optical_depth
wavelengths_per_line_A = lp.velocities_to_line_centered_wavelengths(velocities_kms, line_bundle, a_dagger=a_dagger, a_obs=a_obs)
velocities_per_line_kms = lp.reference_velocities_to_line_centered(velocities_kms, line_bundle, reference_line_idx=0)
tau.wrap_to_ref_line()
optical_depth_wrapped = tau.optical_depth
v_min = -jnp.max(line_bundle.delta_v)

all_figures = pt.generate_all_simulation_plots(
    line_bundle, skewer_bundle, fields, L_box, velocities_kms, wavelengths_per_line_A, velocities_per_line_kms,
    optical_depth, optical_depth_wrapped, a_dagger, a_em, a_obs, v_min, v_max, view_elev=13.0, view_azim=-75.0,
    show=True, plot_normalized_velocity_units=False, plot_wrapped_velocity_units=False, plot_cube=True)

## Spectra and overflux of one skewer
These panels show the transmission of each line and the product of the transmissions, in wrapped velocity units (every line moved onto the Lyα frame). Below them are the single-line overfluxes and the total.

In [ ]:
ii_skewer = 0
line_ids = [l.id for l in line_bundle.lines]
line_colors = {l.id: colors[l.species] for l in line_bundle.lines}
v_kms = np.asarray(velocities_kms)
F = np.exp(-np.asarray(optical_depth_wrapped[:, ii_skewer]))
fig, ax = pt.plot_neon_lines(v_kms, dict(zip(line_ids, F)), total=F.prod(axis=0), colors=line_colors,
                             vlines=(0.0, float(v_max)), xlim=(float(v_min) - 100, float(v_max) + 100), ylim=(-0.05, 1.05),
                             xlabel=r"wrapped velocity $v\ [\mathrm{km\,s^{-1}}]$", ylabel="transmission")
plt.show()

overflux = DeltaFieldBuilder(optical_depth_wrapped).build()
fig, ax = pt.plot_neon_lines(v_kms, {lid: overflux.delta_terms[lab][ii_skewer] for lid, lab in zip(line_ids, overflux.labels)},
                             total=overflux.total[ii_skewer], colors=line_colors,
                             xlabel=r"wrapped velocity $v\ [\mathrm{km\,s^{-1}}]$", ylabel=r"overflux $\delta$")
plt.show()

## P1D and ξ1D by line pair and by order
With 9 skewers the statistics are noisy. The point here is the structure of the decomposition:
- **Pair terms per order** (`plot_p1d_by_order`, `plot_xi_by_order`): the raw spectra and correlations of every line pair, one panel per order.
- **Total recovery** (`plot_*_total_recovery`): the total against the exact sum of the weighted terms, with the raw order sums overlaid.
- **Orders superimposed** (`plot_orders_superimposed`): the exact per-order decomposition (`*_total_by_order`). For P the y axis is kP/π, as in the previous version's custom figure.

The Si III–Lyα separation |Δv| ≈ 2271 km/s is marked on the ξ panels.

In [ ]:
delta_v_list = [float(d) for d in np.asarray(line_bundle.delta_v)]
velocity_domain = float(v_kms[-1] - v_kms[0])
stats = OverfluxStatistics1D(overflux, dv).compute_p1d().compute_xi1d()
k = np.asarray(stats.k)
k_label = r"$k\ [\mathrm{s\,km^{-1}}]$"

fig, axes, axes_res, _ = pt.plot_p1d_by_order(
    stats.ordered_pairs, stats.Pk_pair, k, delta_v_list, velocity_domain, tau.N_v, use_log_x=True,
    residual_mode="sum", show_harmonics=False, use_symlog_main=False, xlabel=k_label)
fig, _, _ = pt.plot_p1d_total_recovery(
    stats.Pk_total, stats.Pk_total_from_terms, stats.Pk_pair, k, delta_v_list, velocity_domain, tau.N_v,
    show_harmonics=False, show_order_sums=True, show_order_sum_residuals=True, main_symlog_linthresh=1.0,
    use_symlog_residual=False, xlabel=k_label)
fig, axs = pt.plot_orders_superimposed(stats, "P", k_weighted=True, yscale="linear")
axs[0].set_xlim(right=0.1)
plt.show()

In [ ]:
lags = np.asarray(stats.lags)
lag_label = r"$\Delta v\ [\mathrm{km\,s^{-1}}]$"
fig, axes, axes_res, _, _ = pt.plot_xi_by_order(
    stats.ordered_pairs, stats.xi_pair, lags, delta_v_list, velocity_domain, residual_mode="sum",
    use_symlog_main=False, show_half_line=False, xlim=(0.0, 3000.0), xlabel=lag_label)
fig, _, _, _ = pt.plot_xi_total_recovery(
    stats.xi_total, stats.xi_total_from_terms, stats.xi_pair, lags, delta_v_list, velocity_domain,
    use_symlog_main=False, use_symlog_residual=False, show_half_line=False, annotate_half_line=False,
    xlim=(0.0, 3000.0), xlabel=lag_label)
fig, axs = pt.plot_orders_superimposed(stats, "xi", delta_v_list=delta_v_list, max_lag=3000.0)
plt.show()

## Input vs recovered
The level of each comparison determines what has to match.

**1. GRF level: exact in expectation.**
- **What is compared:** the density fields come from realisation 0 of `grf`. Here `N_REAL` realisations of the same `grf` (realisation 0 included) are measured directly, as δ itself rather than the normalised δ/σ that the sampler uses. They are compared with `grf.power` and with its discrete inverse transform ξ_in = irfftn(P_in)/dV.
- **Estimators:** `OverfluxStatistics3D` computes P̂ = (dV/N)|rfftn δ|² and ξ̂ = irfftn(P̂)/dV with no window and no smoothing.
- **Binning:** binned values are mode-weighted means, compared with the input binned the same way (`bin_p3d_kmod(…, Pk3d_input=P_in)`).
- **Errors:** for P they are Gaussian, σ = P √(2/(n_modes N_REAL)); for ξ they come from the scatter over realisations.

**2. Physical flux level (the spectra and statistics above): no input match is expected, even for small fluctuations.** Between δ and δ_F the pipeline applies:
- the local density map n = 10^(log_mid + log_amp tanh(δ/σ/1.2)), which is non-linear;
- trilinear interpolation of the GRF grid onto the skewer points, a window that also aliases (the spikes at 2π/Δx in notebook 9);
- thermal Doppler broadening, a Gaussian line profile with b = √(2k_BT/m), which filters τ by e^{−k²b²/4} (`optical_depth/line_profile.py`; notes `sec_flux.tex`, Eq. line_profile);
- the a⁻² factor in dτ/dx and the light-cone mapping v(x) = c ln(a†/a(x)), which together add a slow trend along the skewer;
- `wrap_to_ref_line` for the non-reference lines, a linear interpolation by a non-integer pixel shift that suppresses power by 1 − 2f(1−f)(1 − cos k dv);
- F = e^{−τ} and the normalisation by ⟨F⟩, with τ ≳ 1.

**3. Flux level, linear regime: exact to first order.** This isolates the flux mapping. A twin GRF with the same k₀, n_s and grid, but σ_δ = 0.01, gives τ = τ₀ e^{δ} along the z-lines of the grid, with no interpolation and no thermal kernel.
- To first order (notes `sec_flux.tex`, Eqs. flux and delta_j), δ_F = F/F̄ − 1 = −τ₀ δ + O(δ²), so P1D[δ_F](k_z) → τ₀² P1D_in(k_z).
- P1D_in(k_z) = Σ_{k⊥} P_in(√(k⊥²+k_z²))/(L_x L_y) is the exact discrete projection of `grf_lin.power`.
- `DeltaFieldBuilder` divides each skewer by its own mean, which removes the k_z = 0 mode of every line. The projected input is not zero there, because transverse modes give each line a non-zero mean, so the expected spectrum sets P(k_z = 0) = 0. Without that, the expected ξ1D would be offset by a constant P1D_in(0)/(N_z dz). The other corrections are O(σ_δ²) ≈ 1e-4.

**4. Decomposition:** Σ orders = Σ weighted terms = total is an algebraic identity (shown in the figures above). The notebook runs in float32, so the tolerance is 1e-4.

In [ ]:
N_REAL = 8
as_overflux = lambda d: SimpleNamespace(subset_keys=["a"], delta_terms={"a": d}, C_per_sample=jnp.zeros((d.shape[0], 1)), total=d)
keys3 = [jax.random.PRNGKey(0)] + list(jax.random.split(jax.random.PRNGKey(1), N_REAL - 1))   # 0 = the density field
(Nx, Ny, Nz), (dx3, dy3, dz3) = grf.shape, grf.dx
_, kmod3 = grf.k_grid(real=True)
P3_in = np.asarray(grf.power(kmod3))
xi3_in = np.asarray(xi3d_from_power(P3_in, grf.shape, grf.cell_volume))
ked3 = np.geomspace(0.999 * min(grf.k_min), min(grf.k_max), 13)
P3_list, xi3_list = [], []
for kk in keys3:
    st3 = OverfluxStatistics3D(as_overflux(grf.sample(kk).reshape(Nx * Ny, Nz)), dz3, (dx3, dy3), grid_shape=(Nx, Ny)).compute_xi3d()
    P3_list.append(np.asarray(st3.bin_p3d_kmod(ked3)[0]))
    xi3_list.append(np.asarray(st3.xi3d_total))
_, kb3, _, nb3 = map(np.asarray, st3.bin_p3d_kmod(ked3))
P3_in_b = np.asarray(st3.bin_p3d_kmod(ked3, Pk3d_input=P3_in)[0])
P3_mean, _ = pt.sample_mean_and_error(P3_list)
xi3_mean, xi3_err = pt.sample_mean_and_error(xi3_list)
del xi3_list
fig, axs = pt.plot_p3d_recovery((P3_mean, kb3, ked3, nb3), P3_in_b, power=grf.power, n_real=N_REAL,
                                label=f"GRF, {N_REAL} realisations", units=pt.UNITS_3D_COMOVING)
plt.show()
fig, axs = pt.plot_xi3d_recovery(st3, xi3_in, xi=xi3_mean, sigma=xi3_err, max_lag=30.0, units=pt.UNITS_3D_COMOVING)
plt.show()


In [ ]:
# flux level, linear regime: tau = tau0 exp(delta) along the z-lines of a sigma = 0.01 twin GRF
TAU0 = 1.0
grf_lin = PowerLawGRF(1e-4 * grf.A_s / float(grf.variance()), grf.k_0, grf.n_s, L=grf.L, N=grf.shape)
P1_in = np.asarray(grf_lin.power(kmod3)).sum(axis=(0, 1)) / (grf.L[0] * grf.L[1])        # projected input P1D(k_z)
tau_lin = jnp.concatenate([TAU0 * jnp.exp(grf_lin.sample(kk).reshape(Nx * Ny, Nz)) for kk in keys3])
st_lin = OverfluxStatistics1D(DeltaFieldBuilder(tau_lin[None]).build(), dz3).compute_xi1d()
per_real = lambda a: np.asarray(a).reshape(N_REAL, -1, np.asarray(a).shape[-1]).mean(axis=1)   # mean per box
edges_z = pt.hybrid_k_edges(grf.k_min[2], grf.k_max[2], n_lin=8, n_log=10)
# DeltaFieldBuilder divides each skewer by its own mean, which removes the k_z = 0 mode of every line
P1_exp = TAU0 ** 2 * np.where(np.asarray(st_lin.k) > 0, P1_in, 0.0)
bins_lin = st_lin.bin_p1d(edges_z)
P_lin_exp_b = st_lin.bin_p1d(edges_z, Pk=P1_exp)[0]
_, P_lin_err = pt.sample_mean_and_error([st_lin.bin_p1d(edges_z, Pk=row)[0] for row in per_real(st_lin.Pk_total_per_sample)])
xi_lin_exp = np.fft.fftshift(np.asarray(xi_from_power(P1_exp, Nz, dz3)))
_, xi_lin_err = pt.sample_mean_and_error(per_real(st_lin.xi_total_per_sample))
fig, axs = pt.plot_pk_recovery(bins_lin, P_lin_exp_b, sigma=P_lin_err, label=rf"$\delta_F$, $\sigma_\delta=0.01$, {N_REAL} boxes",
                               k_label=r"$k_z\ [\mathrm{cMpc^{-1}}]$", P_label=r"$P_{\rm 1D}(k_z)\ [\mathrm{cMpc}]$")
axs[0].plot(np.asarray(st_lin.k)[1:], P1_exp[1:], color="k", lw=1.5, label=r"$\tau_0^2\sum_{k_\perp}P_{\rm in}/(L_xL_y)$")
axs[0].legend(fontsize="x-small", frameon=False)
plt.show()
fig, axs = pt.plot_xi_recovery(st_lin, xi_lin_exp, sigma=xi_lin_err, max_lag=40.0, label=r"$\xi[\delta_F]$, linear regime")
axs[1].set_xlabel(r"$\Delta z\ [\mathrm{cMpc}]$")
plt.show()

## Checks

In [ ]:
CHECKS = []
def check(name, passed, detail=""):
    CHECKS.append((name, bool(passed)))
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")
rel = lambda a, b: float(np.max(np.abs(np.asarray(a) - np.asarray(b))) / np.max(np.abs(np.asarray(b))))

same = rel(np.asarray(number_density_sampler.delta_grid) * np.sqrt(float(grf.variance())), grf.sample(keys3[0]))
check("GRF level: the density field is realisation 0 of grf (delta_grid * sigma)", same < 1e-5, f"rel {same:.1e}")
sig3 = pt.pk_sigma(P3_in_b, nb3, N_REAL)
pull3 = (P3_mean - P3_in_b) / sig3
check("GRF level: P3D(|k|) = input (chi2/dof < 2)", np.nanmean(pull3 ** 2) < 2 and np.nanmax(np.abs(pull3)) < 4,
      pt.recovery_summary(P3_mean, P3_in_b, sig3, "P3D"))
c0 = tuple(n // 2 for n in xi3_mean.shape)
dxi = float(np.max(np.abs(xi3_mean - xi3_in)) / xi3_in[c0])
frac = float(np.mean(np.abs(xi3_mean - xi3_in) < 2 * xi3_err))
check("GRF level: xi3D = inverse transform of the input (max |dxi| < 5% xi(0), >= 75% of cells within 2 sigma)",
      dxi < 0.05 and frac >= 0.75, f"max |dxi|/xi(0) = {dxi:.3f}, {100 * frac:.0f}% within 2 sigma ({N_REAL} realisations)")

pull = (np.asarray(bins_lin[0]) - P_lin_exp_b) / P_lin_err
check("flux level, linear: P1D[delta_F] -> tau0^2 x projected P_in (>= 75% within 2 sigma, max |pull| < 6)",
      np.mean(np.abs(pull) < 2) >= 0.75 and np.max(np.abs(pull)) < 6, pt.recovery_summary(bins_lin[0], P_lin_exp_b, P_lin_err, "P1D"))
pull = (np.asarray(st_lin.xi_total) - xi_lin_exp) / xi_lin_err
check("flux level, linear: xi1D[delta_F] = inverse transform of tau0^2 x projected P_in",
      np.mean(np.abs(pull) < 2) >= 0.75 and np.max(np.abs(pull)) < 6, pt.recovery_summary(st_lin.xi_total, xi_lin_exp, xi_lin_err, "xi1D", ratio=False))
P_terms = sum(np.asarray(stats.Pk_total_from_terms_contributions[p]) for p in stats.term_pairs)
check("decomposition, P1D: sum of orders = sum of terms = total",
      max(rel(sum(stats.Pk_total_by_order.values()), stats.Pk_total), rel(P_terms, stats.Pk_total)) < 1e-4,
      f"rel {rel(sum(stats.Pk_total_by_order.values()), stats.Pk_total):.1e}, {rel(P_terms, stats.Pk_total):.1e}")
check("decomposition, xi1D: sum of orders = total", rel(sum(stats.xi_total_by_order.values()), stats.xi_total) < 1e-4,
      f"rel {rel(sum(stats.xi_total_by_order.values()), stats.xi_total):.1e}")
print(f"\nSUMMARY: {sum(p for _, p in CHECKS)}/{len(CHECKS)} checks passed")